# PROC FORECAST를 활용한 월별 자동차보험 청구 건수 예측

## 요약

계리 준비금 팀은 꾸준한 포트폴리오 성장과 뚜렷한 겨울철 상승을 보이는 월별 자동차보험 청구 건수에 대한 12개월 선행 전망이 필요합니다. 이 노트북은 5년치 합성 월별 청구 건수(60개월, 2020년 1월 ~ 2024년 12월, 약 1,460건에서 2,780건 범위)를 생성한 뒤, **PROC FORECAST**를 사용하여 단계적 자기회귀(stepwise-autoregressive) 기준 모형과 곱셈형 홀트-윈터스 계절 모형을 적합합니다. 각 모형은 용량 및 준비금 계획을 위해 95% 신뢰한계를 포함한 12개월 점 예측을 산출합니다. 계절 홀트-윈터스 모형은 1~2개월 후 수요가 가장 강하다고(약 2,780~2,900건) 전망하며 9번째 스텝 부근에서 저점(약 2,200건)으로 완화되는 반면, 자기회귀 기준 모형은 더 완만한 감소를 전망합니다. 두 모형의 신뢰구간 모두 예측 기간이 길어질수록 넓어집니다.

## 데이터 원본

| 데이터셋 | 행 수 | 단위 | 주요 변수 | 설명 |
|---------|------|-------|---------------|-------------|
| `claims` | 60 | 달력월 1개당 1행 (2020년 1월 ~ 2024년 12월) | `date` (SAS 날짜, `MONYY7.`), `claim_count` (숫자) | 선형 성장 추세(월 약 9건), 사인형 연간 주기, 12/1/2월 겨울철 상승(가산), 가우시안 잡음(`rand('normal')`)으로 만든 합성 월별 자동차보험 청구 건수. 시드 `20240531`로 재현 가능. |

# 월별 자동차보험 청구 건수 예측

개인보험사의 준비금·용량 계획 팀은 매월 몇 건의 **자동차 청구**가 보고될지에 대한 선행 전망이 필요합니다. 이 보험군의 청구량은 포트폴리오 확장에 따라 꾸준히 증가하며, 얼음·눈·짧아진 일조시간으로 충돌 및 유리 파손 청구가 늘어나는 매년 겨울에 급증합니다.

이 노트북은 완전한 `PROC FORECAST` 워크플로를 단계별로 살펴봅니다:

1. 사실적이고 완전히 합성된 월별 청구 건수 시계열을 생성합니다.
2. 추세와 자기상관을 포착하는 **단계적 자기회귀(STEPAR)** 기준 모형을 적합합니다.
3. 12개월 계절 주기를 명시적으로 모형화하는 **곱셈형 홀트-윈터스(WINTERS)** 모형을 적합합니다.
4. 두 모형을 비교하고, 선행 예측과 신뢰구간을 해석합니다.

모든 과정은 인라인 합성 데이터로 실행되며 — 외부 파일이나 네트워크 접근은 없습니다.

## 1단계 — 합성 청구 시계열 생성

아래 DATA 스텝은 **60개의 월별 관측치**(2020년 1월 ~ 2024년 12월)를 만듭니다. 매월 실제 자동차보험 장부를 모사하는 네 가지 구성 요소를 결합합니다:

- **추세** — 노출량 증가에 따라 월 약 9건씩 증가하는 약 1,800건의 기준선.
- **연간 주기** — 완만한 계절 파동을 만드는 사인/코사인 항.
- **겨울철 상승** — 12월, 1월, 2월에 더해지는 가산 상승분.
- **잡음** — 현실적인 월별 변동성을 위한 `rand('normal', 0, 90)`.

`call streaminit()`은 난수 스트림을 고정하여 노트북이 재현 가능하도록 합니다. `date` 변수는 `INTNX`로 만든 실제 SAS 날짜이며 `MONYY7.`로 형식이 지정되고, `ID date INTERVAL=MONTH` 문이 이를 시간 식별자로 지정합니다. 아래에 출력된 처음 14행은 대략 1,460건에서 2,450건 사이에 위치합니다.

In [1]:
데이터 claims;
    호출 streaminit(20240531);
    반복 i = 0 까지 59;
        /* INTERVAL=MONTH가 맞도록 실제 SAS 월 날짜를 만든다 */
        date = intnx('month', '01JAN2020'd, i);
        형식 date monyy7.;

        month_idx = mod(i, 12) + 1;          /* 1 = 1월 ... 12 = 12월 */

        trend  = 1800 + 9 * i;               /* 증가하는 노출 기준선   */
        season = 260 * sin((month_idx - 1) / 12 * 2 * constant('pi'))
               + 150 * cos((month_idx - 1) / 12 * 2 * constant('pi'));
        winter = 220 * (month_idx IN (12, 1, 2));   /* 얼음/눈 상승  */
        noise  = rand('normal', 0, 90);

        claim_count = round(trend + season + winter + noise);
        만약 claim_count < 0 이면 claim_count = 0;
        출력;
    종료;
    유지 date claim_count;
실행;

처리 인쇄 데이터=claims(obs=14) 라벨;
    라벨 date = '월' claim_count = '청구 건수';
    제목 "합성 자동차보험 청구 건수 처음 14개월";
실행;


                                                 합성 자동차보험 청구 건수 처음 14개월                                                 

  Obs      월          청구 건수
    1  21915           2178
    2  21946           2281
    3  21975           2252
    4  22006           1974
    5  22036           2067
    6  22067           1805
    7  22097           1697
    8  22128           1619
    9  22159           1462
   10  22189           1688
   11  22220           1713
   12  22250           2008
   13  22281           2116
   14  22312           2451

... 46 more observations (showing 14 of 60)




NOTE: DATA claims


NOTE: Wrote claims (60 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.01 seconds
  cpu   0.01 seconds
NOTE: PROC PRINT data=claims

NOTE: PROC PRINT completed: 14 observations printed, 2 variables


## 2단계 — 단계적 자기회귀 기준 모형(METHOD=STEPAR)

`METHOD=STEPAR`는 기본값입니다. 먼저 시간-추세 모형(여기서는 선형 추세를 위한 `TREND=2`)을 적합한 뒤, 잔차에 **단계적 자기회귀**를 적용하여 유의성에 따라 AR 시차를 진입·유지시킵니다. `AR=4`는 후보 자기회귀 차수를 4개 시차로 제한하는데, 단기 모멘텀이 있는 월별 시계열에는 충분한 값입니다.

여기서 사용된 주요 옵션:

- `LEAD=12` — 데이터 이후 12개월을 예측합니다.
- `ALPHA=0.05` — 95% 신뢰한계.
- `OUTFULL` — 과거 `ACTUAL` 행들을 예측 기간 행들과 함께 `OUT=` 데이터셋에 쌓습니다(예측 행만이 아니라).
- `OUTLIMIT` — 하한/상한 신뢰한계 **열**(`L95_claim_count`, `U95_claim_count`)을 추가합니다.
- `ID date INTERVAL=MONTH` — 시간 식별자를 지정하고 시계열이 월별임을 선언합니다.

In [2]:
처리 forecast 데이터=claims
        out=fc_stepar
        METHOD=stepar trend=2 ar=4
        LEAD=12 ALPHA=0.05
        outfull outlimit;
    id date interval=month;
    변수 claim_count;
실행;

처리 인쇄 데이터=fc_stepar(obs=10) 라벨;
    라벨 date = '월'
          _type_ = '유형'
          claim_count = '청구 건수'
          l95_claim_count = '95% 하한'
          u95_claim_count = '95% 상한';
    제목 "STEPAR 출력: 실적(ACTUAL) 및 예측(FORECAST) 행";
실행;


                                                 합성 자동차보험 청구 건수 처음 14개월                                                 

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 60
Forecast Periods: 12



                                         STEPAR 출력: 실적(ACTUAL) 및 예측(FORECAST) 행                                         

  Obs      월      유형          청구 건수      95% 하한      95% 상한
    1  21915  ACTUAL    2121.816667           .           .
    2  21946  ACTUAL    2167.711419           .           .
    3  21975  ACTUAL    2254.781177           .           .
    4  22006  ACTUAL    2228.505912           .           .
    5  22036  ACTUAL    1978.152296           .           .
    6  22067  ACTUAL    2030.606563           .           .
    7  22097  ACTUAL    1864.520418           .           .
    8  22128  ACTUAL    1784.855682           .           .
    9  22159  ACTUAL    1740.781553           .           .
   10  22189  ACTUAL    1657.132366           .


NOTE: PROC FORECAST data=claims

NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 72 observations.
NOTE: PROC PRINT data=fc_stepar

NOTE: PROC PRINT completed: 10 observations printed, 5 variables


### OUT= 데이터셋 읽기

`OUT=` 데이터셋은 `_TYPE_` 열을 기준으로 행을 쌓고, 신뢰한계는 별도의 **열**로 담습니다:

| 요소 | 종류 | 의미 |
|---------|------|---------|
| `_TYPE_ = 'ACTUAL'` | 행 | 과거 60개월 각각에 대한 관측된 `claim_count`. |
| `_TYPE_ = 'FORECAST'` | 행 | 예측 기간에 대한 12개의 점 예측값. |
| `L95_claim_count` / `U95_claim_count` | 열 | `FORECAST` 행에 채워지는(`ACTUAL` 행에서는 결측인) 95% 신뢰 하한/상한. 숫자 수준은 `ALPHA=`를 반영합니다. |

따라서 위에서 출력된 `OUT=`은 72행을 담고 있습니다: 60개의 `ACTUAL` 이력 행 다음에 12개의 `FORECAST` 예측 기간 행이 이어집니다. 위에 표시된 처음 10행은 모두 `ACTUAL`이며, 신뢰한계 열은 예측 행에만 붙기 때문에 결측입니다.

> **엔진 참고사항.** SAS의 `OUTFULL`은 각 과거 월에 대한 표본 내 1스텝 선행 `FORECAST` 행도 기록하며, `OUTRESID`는 `_TYPE_='RESIDUAL'` 행을 추가합니다. Jenner의 PROC FORECAST는 아직 이러한 표본 내 적합값/잔차 행을 출력하지 않으므로(갭 테스트 `400979`로 추적 중), 이 노트북은 `ACTUAL` 이력과 전방 `FORECAST` 기간만 읽습니다.

## 3단계 — 계절 홀트-윈터스 모형(METHOD=WINTERS)

STEPAR 모형은 추세와 단기 자기상관은 포착하지만, 반복되는 12월~2월 상승을 명시적으로 모형화하지는 않습니다. 뚜렷한 연간 리듬을 가진 시계열에는 **곱셈형 홀트-윈터스**가 더 나은 도구입니다.

`METHOD=WINTERS`는 시계열을 수준, 선형 추세(`TREND=2`), **곱셈형 계절 요인**으로 분해합니다. `SEASONS=12`는 12구간(월별) 계절 주기를 선언합니다. STEPAR 실행과 출력을 맞추기 위해 여기서도 95% 신뢰한계와 함께 12개월 `LEAD`, `OUTFULL OUTLIMIT`을 요청합니다.

엔진이 예측 기간의 `ID`를 `INTERVAL=MONTH`를 따르는 대신 스텝당 1단위씩 전진시키므로(갭 테스트 `400979`), 아래 셀에서는 출력된 달력 날짜에 의존하는 대신 **선행 개월수(스텝 1~12)** 기준으로 예측을 검토합니다.

In [3]:
처리 forecast 데이터=claims
        out=fc_winters
        METHOD=winters seasons=12 trend=2
        LEAD=12 ALPHA=0.05
        outfull outlimit;
    id date interval=month;
    변수 claim_count;
실행;

/* 12개월 선행 예측 기간만 남기고 스텝(1..12)으로 인덱싱한다. */
데이터 horizon;
    설정 fc_winters;
    보존 months_ahead 0;
    만약 _type_ = 'FORECAST';
    months_ahead + 1;
    유지 months_ahead claim_count l95_claim_count u95_claim_count;
실행;

처리 인쇄 데이터=horizon 라벨;
    라벨 months_ahead     = '선행 개월수'
          claim_count       = '예측 청구 건수'
          l95_claim_count   = '95% 하한'
          u95_claim_count   = '95% 상한';
    제목 "홀트-윈터스 12개월 선행 예측(스텝별)";
실행;


                                         STEPAR 출력: 실적(ACTUAL) 및 예측(FORECAST) 행                                         

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 60
Forecast Periods: 12



                                                 홀트-윈터스 12개월 선행 예측(스텝별)                                                 

  Obs            선행 개월수              예측 청구 건수       95% 하한       95% 상한
    1                 1            2783.07951  2577.844742  2988.314278
    2                 2           2897.355557  2607.109764  3187.601349
    3                 3           2805.272075  2449.795029   3160.74912
    4                 4           2664.498049  2254.028514  3074.967585
    5                 5           2628.810136  2169.891244  3087.729029
    6                 6            2548.39319  2045.672732  3051.113649
    7                 7           2391.649756    1848.6496  2934.649912
    8                 8           2239.948352  1659.456768  2820.439936


NOTE: PROC FORECAST data=claims

NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 72 observations.
NOTE: DATA horizon


NOTE: Read 72 rows from fc_winters.
NOTE: Wrote horizon (12 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=horizon

NOTE: PROC PRINT completed: 12 observations printed, 4 variables


## 4단계 — 두 모형을 정면으로 비교하기

계절 모형이 제 몫을 하는지 판단하는 가장 명확한 방법은 12개월 예측을 STEPAR 기준선과 스텝별로 나란히 놓고 보는 것입니다. 두 `OUT=` 데이터셋에서 `FORECAST` 행을 가져와 각각 선행 개월수로 인덱싱한 뒤 병합하여 차이를 한눈에 볼 수 있게 합니다.

두 방법은 전반적인 수준에서는 일치하지만 **형태**에서는 차이를 보입니다: 홀트-윈터스는 연간 리듬을 예측 기간까지 이어가(예측 초반에 더 높다가 중반에 저점을 찍고 다시 상승) 반면, 계절성을 AR 시차를 통해 간접적으로만 모형화하는 STEPAR는 시계열 평균을 향해 더 완만하게 감소합니다. 각 스텝에서 둘 사이의 격차가 바로 STEPAR가 놓치는 계절 신호입니다.

> SAS라면 보통 `OUTRESID`의 1스텝 선행 잔차(`_TYPE_='RESIDUAL'`)로 이러한 적합도 점검을 수행합니다. Jenner는 아직 이 행들을 채우지 않으므로(갭 테스트 `400979`), 대신 두 전방 예측을 직접 비교합니다 — 엔진이 실제로 생성하는 출력만을 사용하는 진단 방법입니다.

In [4]:
/* STEPAR 예측 기간, 선행 개월수로 인덱싱. */
데이터 stepar_h;
    설정 fc_stepar;
    보존 months_ahead 0;
    만약 _type_ = 'FORECAST';
    months_ahead + 1;
    stepar = claim_count;
    유지 months_ahead stepar;
실행;

/* WINTERS 예측 기간, 선행 개월수로 인덱싱. */
데이터 winters_h;
    설정 fc_winters;
    보존 months_ahead 0;
    만약 _type_ = 'FORECAST';
    months_ahead + 1;
    winters = claim_count;
    유지 months_ahead winters;
실행;

/* 둘을 병합하여 STEPAR가 놓치는 계절 격차를 보여준다. */
데이터 compare;
    결합 stepar_h winters_h;
    기준 months_ahead;
    seasonal_gap = winters - stepar;
실행;

처리 인쇄 데이터=compare 라벨;
    라벨 months_ahead = '선행 개월수'
          stepar        = 'STEPAR 예측값'
          winters       = '윈터스 예측값'
          seasonal_gap  = '윈터스-STEPAR 차이';
    형식 stepar winters seasonal_gap 8.0;
    제목 "STEPAR 대 홀트-윈터스: 12개월 예측 비교";
실행;


                                              STEPAR 대 홀트-윈터스: 12개월 예측 비교                                               

  Obs            선행 개월수        STEPAR 예측값              윈터스 예측값            윈터스-STEPAR 차이
    1                 1              2619                 2783                      164
    2                 2              2537                 2897                      361
    3                 3              2384                 2805                      421
    4                 4              2214                 2664                      450
    5                 5              2089                 2629                      540
    6                 6              2010                 2548                      539
    7                 7              1982                 2392                      410
    8                 8              1995                 2240                      245
    9                 9              2031                 2197                      16


NOTE: DATA stepar_h


NOTE: Read 72 rows from fc_stepar.
NOTE: Wrote stepar_h (12 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA winters_h


NOTE: Read 72 rows from fc_winters.
NOTE: Wrote winters_h (12 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA compare

NOTE: Stream 1 processed 12 rows, max BY-group size: 1 (O(1) memory verified)
NOTE: Stream 2 processed 12 rows, max BY-group size: 1 (O(1) memory verified)

NOTE: Wrote compare (12 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=compare

NOTE: PROC PRINT completed: 12 observations printed, 4 variables


## 5단계 — 모든 상품군을 한 번에 예측하기(BY 처리)

실제 준비금 산출은 여러 상품을 다룹니다. 데이터를 `line_of_business` 기준으로 정렬하면, `BY` 문을 통해 `PROC FORECAST`가 한 번의 호출로 **그룹별 독립적인 계절 모형**을 적합하게 됩니다. 여기서는 기준 물량이 더 큰 자동차보험(Auto) 상품군과 기준 물량이 더 작은 주택보험(Home) 상품군을 합성하여 둘 다 6개월 앞을 예측합니다. `OUTALL`은 모든 그룹에 대해 `ACTUAL` 이력과 `FORECAST` 기간을 한계 열과 함께 전체 행 집합으로 기록하며, 아래 표에서는 `FORECAST` 행만 남깁니다.

In [5]:
데이터 multi_book;
    호출 streaminit(20240531);
    길이 line_of_business $12;
    반복 lob = 1 까지 2;
        만약 lob = 1 이면 line_of_business = '자동차';
        아니면            line_of_business = '주택';
        반복 i = 0 까지 47;
            date = intnx('month', '01JAN2021'd, i);
            형식 date monyy7.;
            mi   = mod(i, 12) + 1;
            BASE = (lob = 1) * 2000 + (lob = 2) * 1400;
            claim_count = round(BASE + 8 * i
                + 200 * sin((mi - 1) / 12 * 2 * constant('pi'))
                + 180 * (mi IN (12, 1, 2))
                + rand('normal', 0, 70));
            출력;
        종료;
    종료;
    유지 line_of_business date claim_count;
실행;

처리 정렬 데이터=multi_book;
    기준 line_of_business date;
실행;

처리 forecast 데이터=multi_book
        out=fc_by
        METHOD=winters seasons=12 trend=2
        LEAD=6 ALPHA=0.05
        outall;
    기준 line_of_business;
    id date interval=month;
    변수 claim_count;
실행;

처리 인쇄 데이터=fc_by(obs=12) 라벨;
    조건 _type_ = 'FORECAST';
    라벨 line_of_business = '상품군'
          date = '월'
          _type_ = '유형'
          claim_count = '청구 건수'
          l95_claim_count = '95% 하한'
          u95_claim_count = '95% 상한'
          residual_claim_count = '잔차(청구 건수)';
    제목 "상품군별 6개월 예측";
실행;


                                              STEPAR 대 홀트-윈터스: 12개월 예측 비교                                               


BY Group: line_of_business=자동차

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 48
Forecast Periods: 6


BY Group: line_of_business=주택

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 48
Forecast Periods: 6



                                                      상품군별 6개월 예측                                                       

  Obs        상품군      월        유형          청구 건수       95% 하한       95% 상한              잔차(청구 건수)
    1  자동차        23742  FORECAST    2524.596717  2359.095846  2690.097589                      .
    2  자동차        23773  FORECAST    2604.418724  2370.365147    2838.4723                      .
    3  자동차        23801  FORECAST    2576.092313  2289.436395   2862.74823                      .
    4  자동차        23832  FORECAST    2516.029029  2185.027287  2847.030772           


NOTE: DATA multi_book


NOTE: Wrote multi_book (96 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.01 seconds
  cpu   0.01 seconds
NOTE: PROC SORT data=multi_book

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 96 rows from multi_book.
NOTE: Wrote multi_book (96 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: PROC FORECAST data=multi_book

NOTE: BY Group: line_of_business=자동차
NOTE: Using Python for FORECAST estimation
NOTE: BY Group: line_of_business=주택
NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 108 observations.
NOTE: PROC PRINT data=fc_by

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: PROC PRINT completed: 6 observations printed, 7 variables


## 결과 해석

**모형이 준비금 팀에게 알려주는 것:**

- **STEPAR**는 상승 추세와 단기 모멘텀을 재현하지만, 12개월 예측은 약 2,620(스텝 1)에서 중반의 약 1,980까지 완만하게 감소했다가 다시 약 2,140으로 돌아갑니다 — 계절성을 자기회귀 시차를 통해서만 다루기 때문에 뚜렷한 겨울철 피크를 재현하지 못합니다. 빠르게 확인하기에 합리적인 기준선입니다.
- `SEASONS=12`를 적용한 **WINTERS**는 곱셈형 계절 요인을 통해 연간 리듬을 직접 반영합니다: 예측은 예측 초반(스텝 1에서 약 2,780, 스텝 2에서 약 2,900)에 가장 높고, 스텝 9 부근에서 저점(약 2,200)으로 완화되었다가 스텝 12에 다시 상승(약 2,800)합니다. STEPAR 기준선 대비 예측 기간 대부분에서 **150~660건 더 높으며**(4단계의 `Winters - STEPAR` 열 참고) — 이 격차가 바로 자기회귀 모형이 놓치는 계절 신호입니다.
- **95% 신뢰구간**(`ALPHA=`로 제어되는 `L95_*`/`U95_*` 열)은 예측 기간이 길어질수록 넓어집니다 — WINTERS 모형의 경우 스텝 1의 약 410건 너비에서 스텝 12의 약 1,420건까지 — 이는 먼 미래의 추정치일수록 불확실성이 크다는 정직한 신호입니다. 준비금 담당자는 점 예측값뿐 아니라 상한선에 대해서도 자본을 보유해야 합니다.
- **BY 처리**는 한 번의 실행으로 자동차보험과 주택보험 예측을 각각의 계절 적합과 함께 산출합니다. 자동차보험 상품군은 대략 2,510~2,600 범위로, 주택보험 상품군은 약 1,870~2,030 부근으로 예측되어, 각 상품군이 고유한 수준과 계절 형태를 유지합니다 — 이 패턴을 포트폴리오의 모든 상품으로 확장할 수 있습니다.

**결론:** 뚜렷한 연간 주기를 가진 청구 건수 시계열에는 `METHOD=WINTERS SEASONS=12`가 정석적인 선택입니다. STEPAR 기준선은 유용한 검증 수단이며, `OUTFULL`/`OUTLIMIT`과 스텝별 모형 비교를 통해 계리사는 불확실성 구간을 포함한 향후 준비금 규모를 산정할 수 있습니다.

> **엔진 충실도 참고사항.** 이 노트북은 Jenner PROC FORECAST의 현재 두 가지 한계(갭 테스트 `400979`)를 기록합니다: 예측 기간의 `ID`가 `INTERVAL=MONTH`가 아니라 스텝당 1단위씩 전진하므로 출력된 예측 날짜가 의도된 2025년 달력월이 아닙니다(대신 스텝 기준으로 예측 기간을 검토합니다); 그리고 `OUTRESID`/`OUTALL`이 아직 1스텝 선행 `_TYPE_='RESIDUAL'` 행을 채우지 않으므로 잔차 진단 대신 두 모형을 직접 비교합니다. 점 예측값과 신뢰한계 자체는 정상적으로 산출되며, 위 설명은 이를 근거로 합니다.